## ADP

**ADP** (Aerosol Data Protocol) is a protocol developed by Aerosol d.o.o. for standardized data exchange between instruments and data acquisition systems. It allows for real-time data retrieval, control and monitoring of compatible instruments.

ADP works with **AE36s**, **AE36**, **TCA08** and **TCA09** instruments, while **AE33 is not supported** (AE33 uses legacy AE33 data protocol, which behaves similarly, only commands are different).

**License:** Aerosol Magee Scientific Software License (See LICENSE file for full terms).

In [1]:
from datetime import date, timedelta, datetime
from io import StringIO

import pandas as pd

from aerosol_magee_pytools.data_access.tcp_ip import request_tcp, guess_delimiter

In [2]:
# IP of the instrument - change it to the actual IP address of your AE36s instrument
instrument_ip = '10.10.10.80' # Skylab AE36s
# instrument_ip = '10.10.10.133' # Skylab TCA08
# instrument_ip = '10.10.10.61' # Skylab TCA09

In [3]:
### command INFO
command = '$AERO:INFO\r\n'
received_text = request_tcp(ip=instrument_ip,
                            command=command)
print(received_text)

# parse info response to a dictionary
info = {}
for line in received_text.splitlines():
    line = line.strip()
    if not line or ':' not in line:
        continue
    key, value = line.split(':', 1)
    info[key.strip()] = value.strip()

# there are different fields in INFO response between TCA and AE36, let's synchronize info
if 'Serialnumber' in info:
    info['Serial Number'] = info.pop('Serialnumber' )
    info['Model number'] = info['Serial Number'].split('-')[0]

instrument_typ = info['Model number']
print()
print(f"Instrument type: {instrument_typ}")
print()
print(info)

Connection information
Serialnumber: AE36s-00-00107
Connection ID: 1070
Connection time: 12-Jun-2026 11:15:55
CPU used: 9 % , Memory used: 24 %

Instrument type: AE36s

{'Connection ID': '1070', 'Connection time': '12-Jun-2026 11:15:55', 'CPU used': '9 % , Memory used: 24 %', 'Serial Number': 'AE36s-00-00107', 'Model number': 'AE36s'}


In [4]:
### command LAST return entry from chosen table (table DATA in our case)
command = '$AERO:LAST DATA\r\n'
received_text = request_tcp(ip=instrument_ip,
                            command=command)
print(received_text)

4994325,639168525000000000,639168597000000000,56,1,1,0,0,0,0,1,0,10,10,0,S,874406.5,389357.375,581753.125,878389.375,395728.375,616836.875,855021,415605.875,633416,856660.75,422050.625,607614.125,854540.625,443865.375,606356.75,884508.5,542859.25,713913.375,874904.625,510559.375,632860.75,874626.125,581102,699969.375,873282.125,549354.125,649497.625,51.46946,16.63339,56.65539,18.23596,52.56524,17.04014,45.50778,14.61003,39.76606,12.63679,34.45431,10.86956,32.13602,10.11661,22.70789,7.007157,21.11224,6.530878,338,371,393,394,469,462,392,466,457,424,476,478,400,455,440,405,398,433,392,469,417,392,428,392,402,460,401,0.002709901,0.002582072,0.002720105,0.002498026,0.002290772,0.001888235,0.001862448,-1.710806E-05,-6.916263E-05,19,7.897966,8.527123,7.813732,6.956974,5.778678,5.011155,4.525484,3.045708,2.883677,0.01495743,1.283278,1.113174,1.254372,0.624404,0.4684048,0.2711616,0,0,247,101325,25,3791,1204,4995,2466,158,28.3,30.3,26.8,27.7,29.4,32.5,31.6,1376,291,9


In [5]:
### command FETCH return data from chosen table and time period of last 10 minutes

#### limit time to resent data
end = datetime.now().isoformat(sep=' ', timespec='seconds')
start = (datetime.now() - timedelta(hours=6)).isoformat(sep=' ', timespec='seconds')
###

# command = '$AERO:FETCH DATA "2026-06-11 12:51:00" "2026-06-11 15:51:00"\r\n'
command = f'$AERO:FETCH DATA "{start}" "{end}"\r\n'

received_text = request_tcp(ip=instrument_ip,
                            command=command)
print(received_text)

4993966,639168309600000000,639168381600000000,56,1,1,0,0,0,0,5,0,10,10,0,0,873696.625,528011.375,643124,877001.25,550706.25,688191.25,854707.875,567738.75,702649.5,855854.875,554898.875,663976.875,854250.375,564873.25,655201.375,877486.625,664670,757346.625,874437,621292.5,673299.75,871048.5,666515.875,728256.625,869255.5,623733.75,673413.5,20.92622,6.523033,23.44988,7.131586,21.33621,6.630431,18.0476,5.645178,15.62408,4.855404,13.41321,4.166575,12.45305,3.86912,8.584244,2.635581,7.952047,2.452682,436,488,463,518,579,555,512,557,545,502,473,527,505,502,525,483,475,496,462,461,474,473,505,473,466,459,466,0.002843219,0.002832102,0.002814246,0.002570902,0.002425763,0.001969094,0.001958272,0.00005553583,-0.00015693,12.9,9.309195,10.25047,9.315835,7.660993,6.900806,5.749388,5.132784,3.676252,3.348909,-0.2058115,1.506952,1.22808,0.7777972,0.6794558,0.2661638,-0.002298355,0,0,273,101325,25,3805,1190,4995,2468,156,28.8,33,27,28.7,30,33,32,1376,292,9
4993967,639168310200000000,63916838220000000

In [7]:
# in the end, you can parse data, for example, convert it into pandas dataframe;
# you will need column names first



separator = guess_delimiter(received_text)

df_ae33_data = pd.read_csv(StringIO(received_text.strip()),
                           sep=separator,
                           names=COLUMNS_AE36S_DATA)
print()
print(df_ae33_data)


          ID        TimestampUTC      TimestampLocal  SetupID  G0_Status  \
0    4993966  639168309600000000  639168381600000000       56          1   
1    4993967  639168310200000000  639168382200000000       56          1   
2    4993968  639168310800000000  639168382800000000       56          1   
3    4993969  639168311400000000  639168383400000000       56          1   
4    4993970  639168312000000000  639168384000000000       56          1   
..       ...                 ...                 ...      ...        ...   
355  4994321  639168522600000000  639168594600000000       56          1   
356  4994322  639168523200000000  639168595200000000       56          1   
357  4994323  639168523800000000  639168595800000000       56          1   
358  4994324  639168524400000000  639168596400000000       56          1   
359  4994325  639168525000000000  639168597000000000       56          1   

     G1_Status  G2_Status  G3_Status  G4_Status  G5_Status  G6_Status  \
0            